# Data Viewer — Parquet store to a single HTML table report

Reads whatever `data-loader.ipynb` has written into `store/`, and renders **many instruments
at once into one self-contained HTML file** under `reports/`.

Tables only — no charts. The point is to eyeball the data: what was actually stored, where
the gaps are, and whether the numbers look sane before anything is built on top of them.

## Layout

```
development/marketdata/
├── store/                          <- parquet, written by data-loader.ipynb
├── reports/                        <- HTML output, written here
│   └── market-data.html
├── data-loader.ipynb
└── data-viewer.ipynb               <- this notebook
```

The viewer never imports the loader. It discovers series by **scanning the store directory**,
so there is no duplicated feed code and nothing to keep in sync — the file path *is* the
schema: `store/{asset_class}/{symbol}/{timeframe}/{source}.parquet`.

The generated HTML has no external dependencies: styles and the filter script are inlined, so
the file works offline and can be sent to someone as-is.

In [1]:
# !pip install pandas pyarrow
from __future__ import annotations

import html
import webbrowser
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

STORE_ROOT = Path("store")
REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

OHLCV = ["open", "high", "low", "close", "volume"]

## 1. Discover what is in the store

`scan_store()` walks the parquet tree and reads each file's index only, so the catalogue is
cheap even when the store holds millions of bars.

`gap_pct` is the share of expected bars that are missing, inferred from the median spacing
between timestamps. For daily FX it will sit near 28% because of weekends — that is normal.
A number far above the weekend share on an intraday crypto series means real holes.

In [2]:
def scan_store(root: Path = STORE_ROOT) -> pd.DataFrame:
    """One row per stored series. Path structure supplies the metadata."""
    rows = []
    for path in sorted(Path(root).rglob("*.parquet")):
        asset_class, symbol, timeframe = path.parts[-4:-1]
        idx = pd.read_parquet(path, columns=[]).index
        idx = pd.to_datetime(idx, utc=True)

        gap_pct = np.nan
        if len(idx) > 2:
            step = pd.Series(idx).diff().median()
            if pd.notna(step) and step.total_seconds() > 0:
                expected = (idx.max() - idx.min()) / step + 1
                gap_pct = round(100 * (1 - len(idx) / expected), 1)

        rows.append({
            "asset_class": asset_class, "symbol": symbol, "timeframe": timeframe,
            "source": path.stem, "rows": len(idx),
            "start": idx.min() if len(idx) else pd.NaT,
            "end": idx.max() if len(idx) else pd.NaT,
            "gap_pct": gap_pct,
            "size_kb": round(path.stat().st_size / 1024, 1),
            "path": str(path),
        })

    return pd.DataFrame(rows)


catalogue = scan_store()
catalogue.drop(columns=["path"])

,asset_class,symbol,timeframe,source,rows,start,end,gap_pct,size_kb
0,crypto,BTC-USD,1d,yahoo,616,2025-01-01 00:00:00+00:00,2026-09-08 00:00:00+00:00,0.0,30.8
1,crypto,BTC-USDT,1h,binance,14809,2025-01-01 00:00:00+00:00,2026-09-10 00:00:00+00:00,0.0,649.5
2,crypto,ETH-USDT,1h,binance,14809,2025-01-01 00:00:00+00:00,2026-09-10 00:00:00+00:00,0.0,604.9
3,forex,AUDNZD,1d,yahoo,2003,2019-01-01 00:00:00+00:00,2026-09-09 23:00:00+00:00,28.7,65.1
4,forex,AUDUSD,1d,yahoo,2002,2019-01-01 00:00:00+00:00,2026-09-09 23:00:00+00:00,28.8,70.5
5,forex,EURGBP,1d,yahoo,2003,2019-01-01 00:00:00+00:00,2026-09-09 23:00:00+00:00,28.7,65.5
6,forex,EURJPY,1d,yahoo,2003,2019-01-01 00:00:00+00:00,2026-09-09 23:00:00+00:00,28.7,71.9
7,forex,EURUSD,1d,yahoo,2002,2019-01-01 00:00:00+00:00,2026-09-09 23:00:00+00:00,28.8,68.2
8,forex,GBPUSD,1d,yahoo,2002,2019-01-01 00:00:00+00:00,2026-09-09 23:00:00+00:00,28.8,69.0
9,forex,NZDUSD,1d,yahoo,2002,2019-01-01 00:00:00+00:00,2026-09-09 23:00:00+00:00,28.8,70.6


## 2. Select many series at once

Every filter accepts a single value or a list, and `None` means "no filter". The result is a
slice of the catalogue, which is what the report builder consumes.

In [3]:
def _as_list(value):
    if value is None:
        return None
    return [value] if isinstance(value, str) else list(value)


def select(catalogue: pd.DataFrame, asset_class=None, symbols=None,
           timeframe=None, source=None) -> pd.DataFrame:
    """Filter the catalogue. Each argument takes a string, a list, or None for 'any'."""
    out = catalogue
    for column, wanted in (("asset_class", asset_class), ("symbol", symbols),
                           ("timeframe", timeframe), ("source", source)):
        wanted = _as_list(wanted)
        if wanted is not None:
            out = out[out[column].isin(wanted)]
    return out.reset_index(drop=True)


def load_series(row) -> pd.DataFrame:
    """Read one catalogue row back as an OHLCV frame."""
    df = pd.read_parquet(row["path"])
    df.index = pd.to_datetime(df.index, utc=True)
    df.index.name = "timestamp"
    return df.sort_index()


def series_label(row) -> str:
    return f'{row["symbol"]} · {row["timeframe"]} · {row["source"]}'


def row_label_short(label: str) -> str:
    """Just the symbol — enough for the navigation bar."""
    return label.split(" · ", 1)[0]

## 3. Formatting

Prices need different precision per instrument — `USDJPY` at 153 and `EURUSD` at 1.16 cannot
share a decimal count, and `BTCUSDT` at 60000 needs fewer still. `_decimals_for()` picks the
precision from the median magnitude of the column instead of hard-coding a table of symbols.

In [4]:
def _decimals_for(series: pd.Series) -> int:
    """Choose decimal places from the magnitude of the data."""
    magnitude = pd.to_numeric(series, errors="coerce").abs().median()
    if not np.isfinite(magnitude) or magnitude == 0:
        return 4
    if magnitude >= 1000:
        return 2          # BTCUSDT
    if magnitude >= 100:
        return 3          # USDJPY, EURJPY
    if magnitude >= 0.01:
        return 5          # FX majors and crosses, either side of 1.0
    return 8              # sub-cent crypto


def format_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Return a string frame ready for HTML: aligned decimals, thousands separators."""
    out = pd.DataFrame(index=df.index)
    for name, col in df.items():
        numeric = pd.to_numeric(col, errors="coerce")
        if numeric.isna().all():
            out[name] = col.astype(str)
        elif name == "volume":
            out[name] = numeric.map(lambda v: "" if pd.isna(v) else f"{v:,.0f}")
        else:
            nd = _decimals_for(numeric)
            out[name] = numeric.map(lambda v, nd=nd: "" if pd.isna(v) else f"{v:,.{nd}f}")

    out.index = [t.strftime("%Y-%m-%d %H:%M") if isinstance(t, pd.Timestamp) else str(t)
                 for t in df.index]
    out.index.name = df.index.name or ""
    return out


def table_html(df: pd.DataFrame, index_label: str = "") -> str:
    """Semantic HTML table from an already-formatted frame."""
    head = "".join(f"<th>{html.escape(str(c))}</th>" for c in df.columns)
    header = f"<thead><tr><th class='idx'>{html.escape(index_label)}</th>{head}</tr></thead>"

    body_rows = []
    for idx, row in df.iterrows():
        cells = "".join(f"<td>{html.escape(str(v))}</td>" for v in row)
        body_rows.append(f"<tr><td class='idx'>{html.escape(str(idx))}</td>{cells}</tr>")

    return f"<table>{header}<tbody>{''.join(body_rows)}</tbody></table>"

## 4. The HTML shell

Self-contained: one `<style>` block, one small script providing a live row filter. No CDN, no
fonts to fetch, so the file renders identically offline. It follows the reader's light or dark
system setting.

In [5]:
CSS = """
:root {
  --bg: #ffffff; --fg: #1b1f24; --muted: #6b7280; --line: #e3e6ea;
  --head: #f4f6f8; --zebra: #fafbfc; --accent: #2f6feb;
}
@media (prefers-color-scheme: dark) {
  :root {
    --bg: #0f1419; --fg: #dfe3e8; --muted: #8b949e; --line: #262c34;
    --head: #171d24; --zebra: #131920; --accent: #6ba0ff;
  }
}
* { box-sizing: border-box; }
body {
  margin: 0; padding: 28px 32px 64px; background: var(--bg); color: var(--fg);
  font: 14px/1.5 -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
}
h1 { font-size: 20px; margin: 0 0 4px; }
h2 { font-size: 15px; margin: 34px 0 10px; padding-bottom: 6px; border-bottom: 1px solid var(--line); }
.meta { color: var(--muted); font-size: 12px; margin-bottom: 22px; }
.note { color: var(--muted); font-size: 12px; margin: 6px 0 10px; }
.wrap { overflow-x: auto; border: 1px solid var(--line); border-radius: 6px; }
table { border-collapse: collapse; width: 100%; font-variant-numeric: tabular-nums; }
th, td {
  padding: 5px 10px; text-align: right; white-space: nowrap;
  border-bottom: 1px solid var(--line); font-size: 12.5px;
}
thead th { position: sticky; top: 0; background: var(--head); font-weight: 600; z-index: 1; }
tbody tr:nth-child(even) { background: var(--zebra); }
td.idx, th.idx { text-align: left; color: var(--muted); font-family: ui-monospace, SFMono-Regular, Menlo, monospace; }
tbody tr:hover { background: color-mix(in srgb, var(--accent) 9%, transparent); }
#filter {
  width: 260px; padding: 6px 10px; margin-bottom: 18px; font: inherit; font-size: 13px;
  color: var(--fg); background: var(--bg); border: 1px solid var(--line); border-radius: 6px;
}
nav { margin-bottom: 8px; font-size: 12px; }
nav a { color: var(--accent); text-decoration: none; margin-right: 14px; }
nav a:hover { text-decoration: underline; }
"""

SCRIPT = """
const box = document.getElementById('filter');
box.addEventListener('input', () => {
  const q = box.value.trim().toLowerCase();
  document.querySelectorAll('tbody tr').forEach(tr => {
    tr.style.display = !q || tr.textContent.toLowerCase().includes(q) ? '' : 'none';
  });
});
"""


def html_document(title: str, sections: list[str], subtitle: str = "",
                  nav: list[tuple[str, str]] | None = None) -> str:
    """Assemble the final page. `nav` is a list of (anchor, label) pairs."""
    nav_links = "".join(
        f'<a href="#{html.escape(anchor)}">{html.escape(label)}</a>'
        for anchor, label in (nav or [])
    )

    return (
        "<!doctype html><html lang='en'><head><meta charset='utf-8'>"
        "<meta name='viewport' content='width=device-width, initial-scale=1'>"
        f"<title>{html.escape(title)}</title><style>{CSS}</style></head><body>"
        f"<h1>{html.escape(title)}</h1>"
        f"<div class='meta'>{subtitle}</div>"
        f"<nav>{nav_links}</nav>"
        "<input id='filter' type='search' placeholder='Filter rows…' autocomplete='off'>"
        f"{''.join(sections)}"
        f"<script>{SCRIPT}</script></body></html>"
    )

## 5. Build the report

`build_report()` takes a catalogue slice — **many instruments, one call, one file** — and
writes:

1. **Contents** — every series included, with row counts, coverage and gap share.
2. **Aligned close panel** *(optional)* — one column per instrument on shared timestamps.
   This is the same frame the correlation work consumes, so seeing it here is a direct check
   on the input to that study.
3. **One OHLCV table per instrument**, most recent `max_rows` bars.

Row caps exist because a full hourly series is tens of thousands of rows; the HTML says
plainly when a table was truncated.

In [6]:
def build_report(selection: pd.DataFrame, filename: str = "market-data.html",
                 max_rows: int = 200, include_panel: bool = True,
                 panel_field: str = "close", panel_rows: int = 200,
                 title: str = "Market Data Report") -> Path:
    """Render a catalogue slice into one self-contained HTML file. Returns the path."""
    if selection.empty:
        raise ValueError("selection is empty — nothing to report")

    frames = {series_label(row): load_series(row) for _, row in selection.iterrows()}
    sections: list[str] = []
    nav: list[tuple[str, str]] = []

    # 1. contents
    overview = selection.drop(columns=["path"]).copy()
    for column in ("start", "end"):
        overview[column] = pd.to_datetime(overview[column], utc=True).dt.strftime("%Y-%m-%d %H:%M")
    nav.append(("contents", "Contents"))
    sections.append("<h2 id='contents'>Contents</h2>")
    sections.append(f"<div class='note'>{len(selection)} series from {STORE_ROOT}/</div>")
    sections.append("<div class='wrap'>" + table_html(overview.astype(str), "#") + "</div>")

    # 2. aligned panel across every selected instrument
    if include_panel and len(frames) > 1:
        # When every series shares a timeframe and source, the column header only needs the
        # symbol — repeating "· 1d · yahoo" ten times across the header adds nothing.
        uniform = selection["timeframe"].nunique() == 1 and selection["source"].nunique() == 1
        panel = pd.concat(
            {(row_label_short(label) if uniform else label): df[panel_field]
             for label, df in frames.items()},
            axis=1, join="inner",
        ).dropna(how="all")
        nav.append(("panel", f"Aligned {panel_field}"))
        sections.append(f"<h2 id='panel'>Aligned {panel_field} panel</h2>")
        if panel.empty:
            sections.append("<div class='note'>No timestamps shared by every selected series — "
                            "check that they use the same timeframe.</div>")
        else:
            sections.append(
                f"<div class='note'>{len(panel):,} shared timestamps across {panel.shape[1]} "
                f"instruments; showing the last {min(panel_rows, len(panel)):,}.</div>"
            )
            sections.append("<div class='wrap'>"
                            + table_html(format_frame(panel.tail(panel_rows)), "timestamp")
                            + "</div>")

    # 3. one table per instrument
    for label, df in frames.items():
        anchor = label.replace(" · ", "-").replace(" ", "")
        shown = df.tail(max_rows)
        nav.append((anchor, row_label_short(label)))
        sections.append(f"<h2 id='{html.escape(anchor)}'>{html.escape(label)}</h2>")
        note = f"{len(df):,} bars stored"
        if len(df) > max_rows:
            note += f"; showing the last {max_rows:,}"
        if (df.get("volume", pd.Series(dtype=float)) == 0).all() and len(df):
            note += ". Volume is all zero — expected for Yahoo forex"
        sections.append(f"<div class='note'>{note}.</div>")
        sections.append("<div class='wrap'>" + table_html(format_frame(shown), "timestamp") + "</div>")

    generated = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    subtitle = f"Generated {generated} · {len(selection)} series · source: {STORE_ROOT}/"

    out = REPORTS_DIR / filename
    out.write_text(html_document(title, sections, subtitle, nav), encoding="utf-8")
    print(f"wrote {out.resolve()}  ({out.stat().st_size / 1024:.1f} KB)")
    return out


def open_report(path: Path) -> None:
    webbrowser.open(Path(path).resolve().as_uri())

## 6. Forex — every stored pair, one file

In [7]:
fx_selection = select(catalogue, asset_class="forex", timeframe="1d")

fx_report = build_report(
    fx_selection,
    filename="forex-daily.html",
    title="Forex — daily bars",
    max_rows=120,
    panel_rows=200,
)

fx_selection[["symbol", "timeframe", "source", "rows", "gap_pct"]]

wrote D:\Repository\Quant_Trading\development\marketdata\reports\forex-daily.html  (189.4 KB)


,symbol,timeframe,source,rows,gap_pct
0,AUDNZD,1d,yahoo,2003,28.7
1,AUDUSD,1d,yahoo,2002,28.8
2,EURGBP,1d,yahoo,2003,28.7
3,EURJPY,1d,yahoo,2003,28.7
4,EURUSD,1d,yahoo,2002,28.8
5,GBPUSD,1d,yahoo,2002,28.8
6,NZDUSD,1d,yahoo,2002,28.8
7,USDCAD,1d,yahoo,2002,28.8
8,USDCHF,1d,yahoo,2002,28.8
9,USDJPY,1d,yahoo,2002,28.8


## 7. Crypto — mixed sources and timeframes

The aligned panel is skipped automatically when the selected series share no timestamps, so
mixing a 1h Binance series with a 1d Yahoo series is safe — you just get the per-instrument
tables.

In [ ]:
crypto_report = build_report(
    select(catalogue, asset_class="crypto"),
    filename="crypto.html",
    title="Crypto — stored series",
    max_rows=100,
)

## 8. Everything in the store, one file

In [ ]:
full_report = build_report(
    catalogue,
    filename="market-data.html",
    title="Market Data — full store",
    max_rows=60,
    include_panel=False,     # mixed timeframes share no timestamps
)

# open_report(full_report)

## Notes

- Reports are overwritten by filename. Pass a dated name
  (`f"forex-{pd.Timestamp.utcnow():%Y%m%d}.html"`) if you want to keep a history.
- `reports/*.html` is untracked — the root `.gitignore` does not cover it, so add a rule if
  you would rather not commit generated files.
- The aligned panel needs one timeframe across the selection; that is why the crypto example
  produces per-instrument tables only.
- `gap_pct` near 28% on daily FX is the weekend, not missing data. On intraday crypto it
  should be close to zero — anything larger is a real hole worth investigating before
  backtesting over that period.